# **Phase 3 - Vehicle Classification**

This is an open ended phase. You must build a classifier for the Stanford Cars Dataset. You can use any techniques and knowledge from Phase 1 & 2 to aid you. The ultimate goal is to get as good performance as you can on the test set.

Please follow TA instructions in lab to learn how to access the data.

# **Imports**

Import all the libraries you need for your project. Google Drive mounting code is provided if you plan to use Colab.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import confusion_matrix, classification_report

# **1. Load Dataset**
The Stanford Dataset is provided in ```torchvision.datasets```. However, you must code your own Dataset class to load the dataset.

You will need to create your inherit from ```Dataset``` class to load the dataset and overload the ```___init__()```, ```__len__()```, and ```__getitem__()``` functions.

You will find the [following link](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html) useful to help create the dataset class.

Please also create another function or method in the class: ```show_imgs()``` to help visualize the data. Similar to Phase 1 and 2, you need to display random images and label. Please display 3-5 unique classes.

In [3]:
class CarDataSet(Dataset):

    def __init__(self, csv_file, img_dir, transform=None, class_to_idx=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

        # get classes and map to numbers
        if class_to_idx is None:
            unique = sorted(self.df.iloc[:, -1].unique())
            self.classes = [str(c) for c in unique]
            self.class_to_idx = {c: i for i, c in enumerate(unique)}
        else:
            self.class_to_idx = class_to_idx
            self.classes = [str(k) for k in class_to_idx.keys()]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.iloc[idx, 0])
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        label_name = self.df.iloc[idx, -1]
        label = self.class_to_idx[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label

CarsDataset = CarDataSet

In [4]:
# transforms for train and test
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# load the data
train_data = CarDataSet('train_make.csv', 'images/train', transform=train_transform)
test_data = CarDataSet('test_make.csv', 'images/test', transform=test_transform, class_to_idx=train_data.class_to_idx)

classes = train_data.classes
num_classes = len(classes)

# batch size
batch_size = 32

# import to dataloaders
train_loader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=batch_size, shuffle=False)

FileNotFoundError: [Errno 2] No such file or directory: 'train_make.csv'

In [ ]:
def show_imgs(dataloader, classes):
    images, labels = next(iter(dataloader))
    shown = {}

    for img, lbl in zip(images, labels):
        c = lbl.item()
        if c not in shown:
            shown[c] = img
        if len(shown) == 5:
            break

    fig, axes = plt.subplots(1, len(shown), figsize=(15, 3))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    for ax, (c, img) in zip(axes, shown.items()):
        # unnormalize image for plotting
        img_np = img.numpy().transpose((1, 2, 0))
        img_np = std * img_np + mean
        img_np = np.clip(img_np, 0, 1)

        ax.imshow(img_np)
        ax.set_title(classes[c])
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_imgs(train_loader, classes)

# **2. Define your Model Architecture**

It is up to you to decide which model you use. You can create your own CNN or use transfer learning.

In [ ]:
# model definition
vehicle_classifier = resnet18(weights=ResNet18_Weights.DEFAULT)
vehicle_classifier.fc = nn.Linear(vehicle_classifier.fc.in_features, num_classes)

# load model onto gpu
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
vehicle_classifier.to(device)

# **3. Define Optimizer, Scheduler, Loss, etc.**

You chose the loss function and optimizer.

In [ ]:
lr = 3e-4
num_epochs = 15

# loss function
loss_function = nn.CrossEntropyLoss()

# optimizer
optimizer = torch.optim.Adam(vehicle_classifier.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# **4. Train your Network**

Feel free to use your `train()` from either Phase 1 or Phase 2, whichever you feel is a better choice to understand training to tune it better. Please make sure to plot your learning curves, for both loss and accuracy! Again, this can be per iterations or epoch depending on what you see fit. You will need to save your model as `.pth` file! [This link](https://pytorch.org/tutorials/beginner/saving_loading_models.html) will be useful.

In [ ]:
def train(model, loss_fn, optimizer, scheduler, train_loader, test_loader, num_epochs, device):
    model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    best_acc = 0.0

    for epoch in range(num_epochs):
        # train
        start = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        scheduler.step()

        train_loss = running_loss / total
        train_acc = 100 * correct / total
        train_time = time.time() - start

        # test
        start = time.time()
        model.eval()
        running_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = loss_fn(outputs, labels)

                running_loss += loss.item() * labels.size(0)
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        test_loss = running_loss / total
        test_acc = 100 * correct / total
        test_time = time.time() - start

        # save best weights
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), 'vehicle_classifier.pth')

        # save and print
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)

        print(f'epoch [{epoch+1}/{num_epochs}]  '
              f'train loss: {train_loss:.4f}  train acc: {train_acc:.2f}%  ({train_time:.1f}s)  |  '
              f'test loss: {test_loss:.4f}  test acc: {test_acc:.2f}%  ({test_time:.1f}s)')

    return history

# train network
history = train(vehicle_classifier, loss_function, optimizer, scheduler, train_loader, test_loader, num_epochs, device)

In [ ]:
# plot learning curves
def plot_learning_curve(history, title=''):
    epochs = range(1, len(history['train_loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # loss
    ax1.plot(epochs, history['train_loss'], marker='o', label='train')
    ax1.plot(epochs, history['test_loss'], marker='o', label='test')
    ax1.set_xlabel('epoch')
    ax1.set_ylabel('loss')
    ax1.set_title('loss')
    ax1.legend()
    ax1.grid(True)

    # accuracy
    ax2.plot(epochs, history['train_acc'], marker='o', label='train')
    ax2.plot(epochs, history['test_acc'], marker='o', label='test')
    ax2.set_xlabel('epoch')
    ax2.set_ylabel('accuracy (%)')
    ax2.set_title('accuracy')
    ax2.legend()
    ax2.grid(True)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_learning_curve(history, 'resnet18')

# **5. Evaluate**

Now we will evaluate on validation set. The TA's will run your model on the hold-out test set for your final grade.

## **5.1 Average and Per-class Accuracy**

Feel free to use `test_accuracy()` here to get accuracy. You also must show per-class accuracy. You can write your own accuracy function to get per class accuracy or use `classification_report` from `sklearn`.

In [ ]:
# get predictions
vehicle_classifier.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = vehicle_classifier(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# average accuracy
acc = 100 * np.sum(all_preds == all_labels) / len(all_labels)
print(f'average accuracy: {acc:.2f}%\n')

# per class accuracy
for i, c in enumerate(classes):
    mask = (all_labels == i)
    if np.sum(mask) > 0:
        class_acc = 100 * np.sum(all_preds[mask] == i) / np.sum(mask)
        print(f'{c}: {class_acc:.2f}%')

print('\nclassification report:')
print(classification_report(all_labels, all_preds, target_names=classes))

## **5.2 Confusion Matrix**

Please plot the confusion matrix with all classess. Please make this visually readable.

In [ ]:
# plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(np.arange(len(classes)))
ax.set_yticks(np.arange(len(classes)))
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.set_yticklabels(classes)
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('confusion matrix')

plt.tight_layout()
plt.show()

## **5.3 Check Model Inference**

Please write a function, `show_predictions()` which will show:


1.   The top 5 confidenct predicted images with an example each.
2.   The bottom 5 confident predicted images, with an example each.

For each image, show the true label, what your model predicted and with what confidence.



In [ ]:
def show_predictions(model, dataloader, classes, device):
    model.eval()
    results = []

    with torch.no_grad():
        for images, labels in dataloader:
            outputs = model(images.to(device))
            probs = torch.softmax(outputs, dim=1)
            confs, preds = torch.max(probs, dim=1)

            for i in range(len(images)):
                results.append((confs[i].item(), images[i], labels[i].item(), preds[i].item()))

    # sort by confidence
    results.sort(key=lambda x: x[0], reverse=True)
    top5 = results[:5]
    bot5 = results[-5:]

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    # plot top 5
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for ax, (conf, img, true_l, pred_l) in zip(axes, top5):
        img_np = img.numpy().transpose((1, 2, 0))
        img_np = np.clip(std * img_np + mean, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f'true: {classes[true_l]}\npred: {classes[pred_l]}\nconf: {conf*100:.1f}%')
        ax.axis('off')
    plt.suptitle('top 5 confident predictions')
    plt.tight_layout()
    plt.show()

    # plot bottom 5
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for ax, (conf, img, true_l, pred_l) in zip(axes, bot5):
        img_np = img.numpy().transpose((1, 2, 0))
        img_np = np.clip(std * img_np + mean, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f'true: {classes[true_l]}\npred: {classes[pred_l]}\nconf: {conf*100:.1f}%')
        ax.axis('off')
    plt.suptitle('bottom 5 confident predictions')
    plt.tight_layout()
    plt.show()

show_predictions(vehicle_classifier, test_loader, classes, device)

## **6. Report**

Please write a report discussing all your choices and procedure to implement your vehicle classifier. In your report, include all your choices (i.e. hyperparameters, lr, models, loss, optimizer) and explain why you made those choices to achieve your performance. Your report must be thorough and comprehensive, please discuss fully how you were able to obtain a high performance. Please also include an analysis of your model, whether it may be overfitting or underfitting or fitting well and what you may have done to further improve your performance.

YOUR ANSWER HERE

# **7. Loading your model code**

Please provide code to initialize your model, and code to load the model. Please call your model `vehicle_classifier`. This is so we can easily run your model.

Please also provide here your CarDataSet class and loader with option to add a path for the test data images and csv annotations.

You do not need to do anythig else, this code will be used by TAs to load and run your model to grade on test set.

[This link](https://pytorch.org/tutorials/beginner/saving_loading_models.html) will be useful.

In [ ]:
# paths
test_images_path = 'images/test'
test_annotations_path = 'test_make.csv'
weights_path = 'vehicle_classifier.pth'

test_dataset = CarsDataset(test_annotations_path, test_images_path, transform=test_transform, class_to_idx=train_data.class_to_idx)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

# define model
vehicle_classifier = resnet18(weights=None)
vehicle_classifier.fc = nn.Linear(vehicle_classifier.fc.in_features, len(test_dataset.classes))

# load your weights
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
vehicle_classifier.load_state_dict(torch.load(weights_path, map_location=device))
vehicle_classifier.to(device)
vehicle_classifier.eval()

___
___
# End of Phase 3 😊🥳